# 01 Synthetic VoI Engine V0.1 Validation & Diagnostic Pass

**Academic Project**: Information-Centric Uncertainty-Aware Edge Intelligence for Semantic Sensor Communication  
**Module**: Value of Information (VoI) Engine Version 0.1 Validation Pass  

---  
### Overview & Objectives
This notebook presents the complete diagnostic and stress-testing analysis for the baseline **VoI Engine Version 0.1**.

**Baseline Formulation**: $\text{VoI}_{\text{raw}} = 0.20 N + 0.20 U + 0.20 R + 0.20 T - 0.20 C$  
**Provisional Thresholds**: DISCARD $[0, 0.25)$, BUFFER $[0.25, 0.50)$, SUMMARY $[0.50, 0.70)$, TRANSMIT $[0.70, 1.00]$

### Diagnostic Tests Performed:
1. **Decision Reachability Analysis**
2. **Scenario-Level Breakdown & Decision Distributions**
3. **Monotonicity Verification** for N, U, R, T, C
4. **High Novelty vs Task Relevance** Interaction
5. **Resource Cost Sensitivity**
6. **Weight Sensitivity Analysis** (Configs A, B, C, D)
7. **Threshold Sensitivity Analysis** (Configs A, B, C)
8. **Raw vs Clipped Score Analysis**
9. **Input Correlation Analysis**
10. **Validation Summary Table**

In [ ]:
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Ensure project root is in Python path
sys.path.append(os.path.abspath('..'))

from src.data_generation.synthetic_generator import generate_synthetic_dataset
from src.voi.voi_engine import VoIEngine
from src.voi.decision_policy import DecisionAction
from src.evaluation import compute_summary_statistics
from src.evaluation.diagnostics import (
    run_decision_reachability_analysis,
    run_scenario_analysis,
    plot_scenario_decisions,
    run_monotonicity_experiments,
    run_high_novelty_low_relevance_test,
    run_cost_sensitivity_experiment,
    run_weight_sensitivity_experiment,
    run_clipping_analysis,
    run_threshold_sensitivity_experiment,
    run_correlation_analysis,
    generate_v01_validation_summary
)

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

## 1. Dataset Generation & Processing
Generating 1,000 synthetic observations across 6 behavioral scenarios with seed `42`.

In [ ]:
dataset_path = '../data/synthetic/synthetic_voi_dataset.csv'
df_raw = generate_synthetic_dataset(output_path=dataset_path, num_observations=1000, seed=42)
engine = VoIEngine()
results_df = engine.compute_batch(df_raw)
print(f"Processed {len(results_df)} observations.")
results_df.head(5)

## 2. Decision Reachability Analysis
Evaluating the reachable range of VoI scores under baseline equal weights ($0.20$) and synthetic dataset.

In [ ]:
reachability_df = run_decision_reachability_analysis(results_df, fig_dir='../results/figures', table_dir='../results/tables')
reachability_df

## 3. Scenario-Level Breakdown & Decision Distribution

In [ ]:
sc_df = run_scenario_analysis(results_df, table_dir='../results/tables')
plot_scenario_decisions(results_df, fig_dir='../results/figures')
sc_df

## 4. Single-Variable Monotonicity Sweeps

In [ ]:
run_monotonicity_experiments(engine, fig_dir='../results/figures')
print("Monotonicity sweeps for N, U, R, T, C successfully executed and plotted.")

## 5. High Novelty vs Task Relevance Experiment

In [ ]:
novelty_rel_df = run_high_novelty_low_relevance_test(engine, table_dir='../results/tables')
novelty_rel_df

## 6. Resource Cost Sensitivity Analysis

In [ ]:
cost_df = run_cost_sensitivity_experiment(engine, table_dir='../results/tables')
cost_df

## 7. Weight Sensitivity Analysis

In [ ]:
weight_df = run_weight_sensitivity_experiment(df_raw, fig_dir='../results/figures', table_dir='../results/tables')
weight_df

## 8. Threshold Sensitivity Analysis

In [ ]:
thresh_df = run_threshold_sensitivity_experiment(results_df, table_dir='../results/tables')
thresh_df

## 9. Clipping Analysis & Correlations

In [ ]:
clipping_df = run_clipping_analysis(results_df, table_dir='../results/tables')
corrs = run_correlation_analysis(results_df)
print("=== Correlations with VoI ===")
for k, v in corrs.items():
    print(f"{k}: {v}")
clipping_df

## 10. Final V0.1 Validation Summary Table

In [ ]:
summary_v01 = generate_v01_validation_summary(table_dir='../results/tables')
summary_v01